In [1]:
# Financial Statement Extractor - Lab Notebook
# Setup and Imports

import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd
from edgar import Company, set_identity
from typing import Optional, Literal
import asyncio

# Load environment variables
load_dotenv()

# Set SEC identity
sec_identity = os.getenv("SEC_ID")
if not sec_identity:
    raise ValueError("SEC_ID not found in .env file")
set_identity(sec_identity)

# Import mapping to verify it loads
from income_statement_xbrl_mapping import INCOME_STATEMENT_MAPPING
print(f"✓ Loaded XBRL mapping: {len(INCOME_STATEMENT_MAPPING)} fields")

# Import helper functions from the extractor
from fin_st_extractor_updated import (
    get_filing,
    extract_income_statement,
    format_for_display,
    validate_income_statement,
    print_validation_results
)

/home/pedro/projects/fin_import2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Loaded XBRL mapping: 30 fields
✓ Loaded comprehensive XBRL mapping (148 concepts, 30 fields)


In [ ]:
# Configuration

TICKER = "PFE"
FILING_TYPE = "10-K"
YEAR = 2024
QUARTER = None
USE_AI_FALLBACK = True

print("="*80)
print("FINANCIAL STATEMENT EXTRACTOR")
print("="*80)
print(f"\nConfiguration:")
print(f"  Ticker: {TICKER}")
print(f"  Filing Type: {FILING_TYPE}")
print(f"  Year: {YEAR if YEAR else 'Most Recent'}")
if FILING_TYPE == "10-Q":
    print(f"  Quarter: Q{QUARTER if QUARTER else 'Most Recent'}")
print(f"  AI Fallback: {'Enabled' if USE_AI_FALLBACK else 'Disabled'}")
print()

FINANCIAL STATEMENT EXTRACTOR

Configuration:
  Ticker: COHU
  Filing Type: 10-K
  Year: 2024
  AI Fallback: Enabled



In [3]:
# Get Filing and Extract Income Statement

filing = get_filing(
    ticker=TICKER,
    filing_type=FILING_TYPE,
    year=YEAR,
    quarter=QUARTER
)

income_statement_df = await extract_income_statement(
    filing=filing,
    ticker=TICKER,
    filing_type=FILING_TYPE,
    year=YEAR,
    quarter=QUARTER,
    use_ai_fallback=USE_AI_FALLBACK
)

formatted_df = format_for_display(income_statement_df)

✓ Retrieved company: COHU INC (COHU)
✓ Retrieved 10-K filing
  Filing Date: 2025-02-20
  Period of Report: 2024-12-28

EXTRACTING INCOME STATEMENT

✓ Available periods: 2024-12-28, 2023-12-30, 2022-12-31
✓ Using most recent period: 2024-12-28
✓ AI fallback enabled for unmapped concepts

Extracting 30 line items...
  → AI fallback for gross_profit (predefined concepts not found)...
  → Checking 2 unmapped concepts with AI...
Mapper called for concept:  InvestmentIncomeInterestAndDividend
Mapper called for concept:  GainsLossesOnExtinguishmentOfDebt
  ✗ AI found no matching concepts for gross_profit
  → AI fallback for other_operating_expenses (predefined concepts not found)...
  → Checking 2 unmapped concepts with AI...
Error mapping concept InvestmentIncomeInterestAndDividend: Max turns (10) exceeded
Mapper called for concept:  GainsLossesOnExtinguishmentOfDebt
  ✗ AI found no matching concepts for other_operating_expenses
  → AI fallback for interest_income (predefined concepts not fo

In [4]:
# Display Results

print("\n" + "="*80)
print(f"INCOME STATEMENT - {TICKER}")
print("="*80)

if not income_statement_df.empty:
    metadata = income_statement_df.iloc[0]
    print(f"\nFiling Metadata:")
    print(f"  Ticker:           {metadata['Ticker']}")
    print(f"  Fiscal Year:      {metadata['Fiscal_Year']}")
    print(f"  Period End Date:  {metadata['Period_End_Date']}")
    print(f"  Filing Date:      {metadata['Filing_Date']}")
    print(f"  Filing Type:      {metadata['Filing_Type']}")
    print(f"  Period Type:      {metadata['Period_Type']}")
    if metadata['Quarter']:
        print(f"  Quarter:          Q{metadata['Quarter']}")

print("\n" + "-"*80)
print("FINANCIAL DATA:")
print("-"*80)

print("\nREVENUE & COSTS:")
revenue_section = formatted_df[
    formatted_df['Field'].isin(['revenue', 'cost_of_revenue', 'gross_profit'])
]
print(revenue_section[['Status', 'Field', 'Value', 'Concept']].to_string(index=False))

print("\n\nOPERATING EXPENSES:")
opex_section = formatted_df[
    formatted_df['Field'].isin([
        'research_development', 'selling_general_admin',
        'depreciation_amortization', 'restructuring_charges',
        'other_operating_expenses', 'total_operating_expenses',
        'operating_income'
    ])
]
print(opex_section[['Status', 'Field', 'Value', 'Concept']].to_string(index=False))

print("\n\nNON-OPERATING ITEMS:")
nonop_section = formatted_df[
    formatted_df['Field'].isin([
        'interest_income', 'interest_expense',
        'equity_method_investments', 'investment_gains_losses',
        'other_nonoperating_income'
    ])
]
print(nonop_section[['Status', 'Field', 'Value', 'Concept']].to_string(index=False))

print("\n\nNET INCOME:")
netincome_section = formatted_df[
    formatted_df['Field'].isin([
        'pretax_income', 'income_tax_expense',
        'net_income_continuing_ops', 'discontinued_operations',
        'net_income', 'net_income_attributable_to_nci',
        'net_income_attributable_to_parent'
    ])
]
print(netincome_section[['Status', 'Field', 'Value', 'Concept']].to_string(index=False))

print("\n\nPER SHARE DATA:")
pershare_section = formatted_df[
    formatted_df['Field'].isin([
        'basic_eps', 'diluted_eps',
        'basic_shares', 'diluted_shares'
    ])
]
print(pershare_section[['Status', 'Field', 'Value', 'Concept']].to_string(index=False))


INCOME STATEMENT - COHU

Filing Metadata:
  Ticker:           COHU
  Fiscal Year:      2024
  Period End Date:  2024-12-28
  Filing Date:      2025-02-20
  Filing Type:      10-K
  Period Type:      Annual

--------------------------------------------------------------------------------
FINANCIAL DATA:
--------------------------------------------------------------------------------

REVENUE & COSTS:
Status           Field        Value                                                            Concept
     ✓         revenue $401,779,000                RevenueFromContractWithCustomerIncludingAssessedTax
     ✓ cost_of_revenue $221,485,000 CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization
     ✗    gross_profit    Not Found                                                          Not Found


OPERATING EXPENSES:
Status                     Field        Value                                Concept
     ✓      research_development  $84,797,000          ResearchAndDevelopment

In [5]:
# Validate

validations = validate_income_statement(income_statement_df)
print_validation_results(validations)


VALIDATION CHECKS

✓ PASS - Tax Logical
  net_income: $-69,818,000
  pretax_income: $-64,946,000

✓ All validation checks passed!


In [6]:
# Export to CSV

fiscal_year = income_statement_df.iloc[0]['Fiscal_Year']
filing_type_clean = income_statement_df.iloc[0]['Filing_Type'].replace('/', '-')
quarter_str = f"_Q{income_statement_df.iloc[0]['Quarter']}" if income_statement_df.iloc[0]['Quarter'] else ""

os.makedirs('./reports', exist_ok=True)

export_path = f"./reports/{TICKER}_{filing_type_clean}_{fiscal_year}{quarter_str}_income_statement.csv"
income_statement_df.to_csv(export_path, index=False)
print(f"\n✓ Data exported to: {export_path}")

found = (income_statement_df['Value'].notna()).sum()
total = len(income_statement_df)
print(f"\nData Quality Summary:")
print(f"  Fields extracted: {found}/{total} ({found/total*100:.1f}%)")
print(f"  File size: {os.path.getsize(export_path):,} bytes")


✓ Data exported to: ./reports/COHU_10-K_2024_income_statement.csv

Data Quality Summary:
  Fields extracted: 18/30 (60.0%)
  File size: 3,160 bytes
